In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_rows', None)

In [74]:
X_train = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/predictors_train_filtered.csv')
X_test = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/predictors_test_filtered.csv')
y_train = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/target_train_filtered_logariphmed.csv')
y_test = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/target_test_filtered_logariphmed.csv')

In [16]:
train_players = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/train_players_names.csv')
test_players = pd.read_csv('csv/final_datasets/for_linear_model/MF,FW/test_players_names.csv')

In [6]:
all_cols = X_train.columns
all_cols

Index(['Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK',
       'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK',
       '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'OG', 'Sh', 'SoT',
       'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1'],
      dtype='object')

<b>Регрессор</b>

In [92]:
LR_regressor = LinearRegression()
LR_regressor.fit(X_train, y_train)

LinearRegression()

<b>Предсказанные значения</b>

In [93]:
y_pred_log = LR_regressor.predict(X_test)
y_pred = np.exp(y_pred_log)
y_test_original = np.exp(y_test)

y_train_pred = np.exp(LR_regressor.predict(X_train))
y_train_original = np.exp(y_train)

<b>Метрики</b>

In [94]:
mae_test = mean_absolute_error(y_test_original, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test_original, y_pred))
r2_test = r2_score(y_test_original, y_pred)
mape_test = mean_absolute_percentage_error(y_test_original, y_pred)

mae_train = mean_absolute_error(y_train_original, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train_original, y_train_pred))
r2_train = r2_score(y_train_original, y_train_pred)
mape_train = mean_absolute_percentage_error(y_train_original, y_train_pred)

metrics = pd.DataFrame({
    'MAE':[mae_test, mae_train],
    'RMSE':[rmse_test, rmse_train],
    'MAPE':[mape_test, mape_train],
    'R2':[r2_test, r2_train]
}, index = ['test', 'train'])

<b>Сравнение остатков</b>

In [95]:
y_pred = pd.Series(np.reshape(y_pred, len(y_pred)), name='Value_pred')
y_train_pred = pd.Series(np.reshape(y_train_pred, len(y_train_pred)), name='Value_pred')

comp_leftovers_test = pd.concat([test_players, y_pred, y_test_original], axis=1)
comp_leftovers_train = pd.concat([train_players, y_train_pred, y_train_original], axis=1)

<b>Коэффициенты</b>

In [96]:
coeffs = LR_regressor.coef_[0]
coeffs_df = pd.DataFrame({
    'coeff':coeffs.astype('float')
}, index=X_train.columns)
coeffs_df = coeffs_df.sort_values(by='coeff', key=abs, ascending=False)

<b>Все переменные</b>

In [13]:
coeffs_df

,coeff
Min,4.304878
90s,-4.013622
G-PK.1,-1.929223
Gls.1,1.747549
league_GB1,0.835877
G+A-PK,0.730689
league_L1,0.596664
league_ES1,0.521720
SoT,0.473778
league_IT1,0.454867


In [14]:
metrics

,MAE,RMSE,MAPE,R2
test,6.262792,12.732324,0.979577,0.054685
train,5.132817,12.021404,0.721469,0.463513


In [19]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,3.676679,30.00
1,Dani Rodriguez,3.736283,0.80
2,Gustavo Klismahn,1.371311,0.60
3,Zico Buurmeester,1.525448,3.50
4,Elves Balde,1.025636,0.30
5,Hakon Arnar Haraldsson,10.837656,18.00
6,Emmanuel Ekong,1.392482,0.55
7,Simone Verdi,0.587379,1.50
8,Umut Tohumcu,5.928324,6.00
9,Ethan Mbappe,1.493469,5.00


In [20]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,5.114546,12.000
1,Pau Cabanes,3.108334,0.800
2,Unai Gomez,7.460736,6.000
3,Danila Kozlov,1.150891,2.000
4,Gerard Vandeplas,0.539660,0.400
5,Shavy Babicka,3.218635,2.500
6,Nikola Cumic,0.935699,1.500
7,Nicola Sansone,0.467558,0.400
8,Francisco Trincao,35.709630,35.000
9,Mehdi Merghem,0.422012,0.450


<b>Модель 1</b>

In [22]:
X_train = X_train[['G+A', 'G+A.1', 'Sh', 'Ast.1', 'Sh/90', 'SoT/90', '90s', 'Fld', 'G/SoT', 'TklW', 'Crs', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Sh', 'Ast.1', 'Sh/90', 'SoT/90', '90s', 'Fld', 'G/SoT', 'TklW', 'Crs', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [28]:
coeffs_df

,coeff
league_GB1,0.751768
league_L1,0.510106
league_ES1,0.421669
league_FR1,0.411759
league_IT1,0.408176
Age,-0.327147
Sh,0.323125
Ast.1,0.304993
90s,0.266206
G/SoT,0.228467


In [29]:
metrics

,MAE,RMSE,MAPE,R2
test,4.828872,9.216708,0.867970,0.504649
train,5.497479,12.940978,0.763351,0.378297


In [30]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,3.521731,30.00
1,Dani Rodriguez,4.685114,0.80
2,Gustavo Klismahn,1.341554,0.60
3,Zico Buurmeester,1.553045,3.50
4,Elves Balde,1.364235,0.30
5,Hakon Arnar Haraldsson,10.111187,18.00
6,Emmanuel Ekong,1.736135,0.55
7,Simone Verdi,0.625940,1.50
8,Umut Tohumcu,4.919224,6.00
9,Ethan Mbappe,2.392701,5.00


In [31]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,7.237241,12.000
1,Pau Cabanes,2.968813,0.800
2,Unai Gomez,6.909523,6.000
3,Danila Kozlov,1.577470,2.000
4,Gerard Vandeplas,0.526407,0.400
5,Shavy Babicka,5.364708,2.500
6,Nikola Cumic,0.959405,1.500
7,Nicola Sansone,0.639685,0.400
8,Francisco Trincao,29.407427,35.000
9,Mehdi Merghem,0.447434,0.450


<b>Модель 2</b>

In [38]:
X_train = X_train[['G+A', 'Sh', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'TklW', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Sh', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'TklW', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [44]:
coeffs_df

,coeff
league_GB1,0.775596
league_L1,0.539960
league_IT1,0.438593
league_FR1,0.432594
league_ES1,0.432063
Age,-0.341519
Sh,0.317403
Ast.1,0.270200
G/SoT,0.214043
90s,0.203485


In [45]:
metrics

,MAE,RMSE,MAPE,R2
test,4.838630,9.025999,0.877333,0.524937
train,5.466847,13.088024,0.768487,0.364089


In [46]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,3.331164,30.00
1,Dani Rodriguez,4.789001,0.80
2,Gustavo Klismahn,1.443584,0.60
3,Zico Buurmeester,1.535796,3.50
4,Elves Balde,1.416420,0.30
5,Hakon Arnar Haraldsson,10.142502,18.00
6,Emmanuel Ekong,1.770281,0.55
7,Simone Verdi,0.678181,1.50
8,Umut Tohumcu,5.097386,6.00
9,Ethan Mbappe,2.402541,5.00


In [47]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,7.141220,12.000
1,Pau Cabanes,3.125264,0.800
2,Unai Gomez,6.225660,6.000
3,Danila Kozlov,1.625387,2.000
4,Gerard Vandeplas,0.513498,0.400
5,Shavy Babicka,5.433185,2.500
6,Nikola Cumic,0.897488,1.500
7,Nicola Sansone,0.620219,0.400
8,Francisco Trincao,24.345933,35.000
9,Mehdi Merghem,0.461010,0.450


<b>Модель 3</b>

In [52]:
X_train = X_train[['G+A', 'G+A.1', 'Sh', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Crs', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Sh', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Crs', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [58]:
coeffs_df

,coeff
league_GB1,0.739391
league_L1,0.491097
league_ES1,0.406008
league_FR1,0.401513
league_IT1,0.388880
90s,0.372766
Age,-0.333867
Ast.1,0.325004
Sh,0.311321
G/SoT,0.239976


In [59]:
metrics

,MAE,RMSE,MAPE,R2
test,4.914413,9.336048,0.841872,0.491738
train,5.481849,13.122266,0.762994,0.360757


In [60]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,3.617585,30.00
1,Dani Rodriguez,4.265362,0.80
2,Gustavo Klismahn,1.312897,0.60
3,Zico Buurmeester,1.518145,3.50
4,Elves Balde,1.424956,0.30
5,Hakon Arnar Haraldsson,10.606293,18.00
6,Emmanuel Ekong,1.694140,0.55
7,Simone Verdi,0.627720,1.50
8,Umut Tohumcu,4.882786,6.00
9,Ethan Mbappe,2.507677,5.00


In [61]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,7.623980,12.000
1,Pau Cabanes,2.934369,0.800
2,Unai Gomez,7.481754,6.000
3,Danila Kozlov,1.579379,2.000
4,Gerard Vandeplas,0.558538,0.400
5,Shavy Babicka,5.421215,2.500
6,Nikola Cumic,1.020080,1.500
7,Nicola Sansone,0.631126,0.400
8,Francisco Trincao,29.339394,35.000
9,Mehdi Merghem,0.413555,0.450


<b>Модель 4</b>

In [62]:
X_train = X_train[['G+A', 'G+A.1', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [68]:
coeffs_df

,coeff
league_GB1,0.745626
league_L1,0.480961
90s,0.443726
league_IT1,0.392746
league_FR1,0.386713
league_ES1,0.372358
Age,-0.351727
Ast.1,0.290474
G/SoT,0.219604
G+A,0.173204


In [69]:
metrics

,MAE,RMSE,MAPE,R2
test,4.868575,9.109763,0.848586,0.516078
train,5.125910,11.648225,0.788811,0.496305


In [70]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,3.543022,30.00
1,Dani Rodriguez,5.100796,0.80
2,Gustavo Klismahn,1.433737,0.60
3,Zico Buurmeester,1.495277,3.50
4,Elves Balde,1.461662,0.30
5,Hakon Arnar Haraldsson,10.972044,18.00
6,Emmanuel Ekong,1.831694,0.55
7,Simone Verdi,0.662893,1.50
8,Umut Tohumcu,4.653413,6.00
9,Ethan Mbappe,3.161541,5.00


In [71]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,7.471218,12.000
1,Pau Cabanes,2.900548,0.800
2,Unai Gomez,6.750610,6.000
3,Danila Kozlov,1.525404,2.000
4,Gerard Vandeplas,0.523412,0.400
5,Shavy Babicka,5.622375,2.500
6,Nikola Cumic,0.886108,1.500
7,Nicola Sansone,0.815571,0.400
8,Francisco Trincao,20.784443,35.000
9,Mehdi Merghem,0.428204,0.450


In [77]:
X_train = X_train[['G+A', 'G+A.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'G+A.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [83]:
coeffs_df

,coeff
league_GB1,0.723024
league_L1,0.469459
90s,0.459986
league_IT1,0.386027
Age,-0.352579
league_FR1,0.339752
league_ES1,0.335996
G+A,0.213450
G+A.1,0.207368
league_NL1,-0.096316


In [84]:
metrics

,MAE,RMSE,MAPE,R2
test,4.716678,8.632656,0.828857,0.56544
train,5.252818,12.757309,0.827967,0.39582


In [87]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,2.679313,30.00
1,Dani Rodriguez,4.077808,0.80
2,Gustavo Klismahn,1.123630,0.60
3,Zico Buurmeester,1.513418,3.50
4,Elves Balde,1.301026,0.30
5,Hakon Arnar Haraldsson,10.572635,18.00
6,Emmanuel Ekong,2.061732,0.55
7,Simone Verdi,0.765563,1.50
8,Umut Tohumcu,4.304514,6.00
9,Ethan Mbappe,2.632617,5.00


In [88]:
comp_leftovers_train

,Player,Value_pred,Value
0,Hwang Hee-chan,7.921569,12.000
1,Pau Cabanes,3.356514,0.800
2,Unai Gomez,5.487338,6.000
3,Danila Kozlov,1.671769,2.000
4,Gerard Vandeplas,0.694201,0.400
5,Shavy Babicka,5.482835,2.500
6,Nikola Cumic,0.957301,1.500
7,Nicola Sansone,0.779500,0.400
8,Francisco Trincao,17.020089,35.000
9,Mehdi Merghem,0.456179,0.450


In [90]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
133,Cole Palmer,220.258500,120.000
193,Matheus Cunha,126.345722,60.000
146,Eberechi Eze,71.394409,55.000
89,Amad Diallo,67.741666,45.000
91,Mikkel Damsgaard,66.172212,28.000
221,Florian Wirtz,58.796214,140.000
66,Alejandro Garnacho,55.245747,45.000
154,Dejan Kulusevski,52.886235,50.000
12,Omari Hutchinson,50.785307,22.000
140,Georginio Rutter,39.235433,32.000


In [91]:
X_train = X_train[['G+A', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]
X_test = X_test[['G+A', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']]

In [97]:
coeffs_df

,coeff
league_GB1,0.735365
league_L1,0.483841
league_IT1,0.399948
90s,0.376520
G+A,0.367518
league_FR1,0.357504
Age,-0.353592
league_ES1,0.343101
G/SoT,0.174225
league_NL1,-0.088982


In [98]:
metrics

,MAE,RMSE,MAPE,R2
test,4.565101,8.330637,0.831055,0.595315
train,5.536570,14.358948,0.837187,0.234591


In [99]:
comp_leftovers_test

,Player,Value_pred,Value
0,Pedro Goncalves,2.267385,30.00
1,Dani Rodriguez,4.099590,0.80
2,Gustavo Klismahn,0.918985,0.60
3,Zico Buurmeester,1.563450,3.50
4,Elves Balde,1.216618,0.30
5,Hakon Arnar Haraldsson,11.004373,18.00
6,Emmanuel Ekong,2.340219,0.55
7,Simone Verdi,0.860571,1.50
8,Umut Tohumcu,3.860250,6.00
9,Ethan Mbappe,3.303771,5.00


In [100]:
comp_leftovers_train.sort_values('Value_pred', ascending=False)

,Player,Value_pred,Value
133,Cole Palmer,257.644664,120.000
193,Matheus Cunha,146.814299,60.000
146,Eberechi Eze,77.224173,55.000
89,Amad Diallo,71.149596,45.000
91,Mikkel Damsgaard,66.147869,28.000
221,Florian Wirtz,60.735551,140.000
66,Alejandro Garnacho,57.277039,45.000
154,Dejan Kulusevski,56.108400,50.000
12,Omari Hutchinson,48.046348,22.000
140,Georginio Rutter,42.256466,32.000


In [72]:
base_cols = ['G+A', 'G+A.1', 'Ast.1', 'Sh/90', '90s', 'Fld', 'G/SoT', 'Age', 'league_ES1', 'league_FR1',
       'league_GB1', 'league_IT1', 'league_L1', 'league_NL1', 'league_PO1',
       'league_RU1']

In [75]:
cols_to_add = [col for col in all_cols if not(col in base_cols)]
new_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_add:
    X_train_new = X_train[base_cols + [col]]
    X_test_new = X_test[base_cols + [col]]
    LR_regressor.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new<mae_test or mape_new<mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=new_cols.columns, index=[col])
        new_cols = pd.concat([new_cols, row])
new_cols

,mae,mape,r2
base,4.868575,0.848586,0.516078
Starts,4.652191,1.065595,-0.266716
Gls,4.834931,1.101207,-0.548908
Ast,4.834931,1.101207,-0.548908
G-PK,4.738015,1.096229,-0.401524
PK,4.828316,1.116326,-0.459347
PKatt,4.854903,1.117976,-0.501586
Fls,4.734653,1.109610,-0.337099
TklW,4.752886,1.110109,-0.440269
OG,4.868440,1.117182,-0.520659


In [76]:
cols_to_delete = [col for col in base_cols if not(col.startswith('league'))]
del_cols = pd.DataFrame({
    'mae':[mae_test],
    'mape':[mape_test],
    'r2':[r2_test]
}, index=['base'])
for col in cols_to_delete:
    X_train_new = X_train[[c for c in base_cols if c != col]]
    X_test_new = X_test[[c for c in base_cols if c != col]]
    LR_regressor.fit(X_train_new, y_train)
    y_pred_new = np.exp(LR_regressor.predict(X_test_new))
    mae_new = mean_absolute_error(y_pred_new, y_test_original)
    mape_new = mean_absolute_percentage_error(y_pred_new, y_test_original)
    r2_new = r2_score(y_pred_new, y_test_original)
    if mae_new < mae_test or mape_new < mape_test or r2_new > r2_test:
        row = pd.DataFrame([[mae_new, mape_new, r2_new]], columns=del_cols.columns, index=[col])
        del_cols = pd.concat([del_cols, row])
del_cols

,mae,mape,r2
base,4.868575,0.848586,0.516078
Ast.1,4.716678,1.236882,-0.469451
Fld,4.854948,1.108384,-0.440092
